# Module 8: Continuous Batching and Saturation

In Module 4 you saw decode is memory-bound, and one request leaves the GPU mostly idle. The fix for that idleness is to run many requests at once. This module drives rising concurrency at your own vLLM and watches continuous batching pack the requests into one running batch, until the GPU fills and the queue forms. You read the saturation point off the metrics, no Grafana, straight from the server.

## Learning objectives
- Drive rising concurrency at your endpoint with one uniform load tool
- Watch throughput climb, flatten at the knee, then stop paying off
- Read `num_requests_waiting`, `kv_cache_usage_perc`, and `num_preemptions_total` to name the bottleneck
- Tie the knee back to the memory-bound limit from Module 4

## Prerequisites
- Finished Module 4, with the memory-bound picture and the ridge point
- A live vLLM endpoint, and `common/loadtest.py` plus `common/ttft_metrics.py`
- About 15 minutes

References: [Anatomy of vLLM](https://blog.vllm.ai/2025/09/05/anatomy-of-vllm.html) &middot; [vLLM production metrics](https://docs.vllm.ai/en/latest/design/metrics/) &middot; [Continuous batching](https://www.anyscale.com/blog/continuous-batching-llm-inference) &middot; [GuideLLM](https://github.com/vllm-project/guidellm)

## Saturation design basics

TODO (Omer): a request does not break a server all at once. It breaks in order, and each stage shows up in a different metric.

- Headroom: the batch grows, throughput climbs, latency barely moves.
- The knee: throughput flattens, per-request latency starts to rise. This is your operating point.
- Queue: `num_requests_waiting` grows. Requests wait before prefill even starts.
- KV pressure: `kv_cache_usage_perc` nears 1.0.
- Preemption: `num_preemptions_total` climbs as the server evicts and recomputes.

![TODO: the five saturation stages, each labeled with the metric that proves it](images/08_saturation_architecture.png)

## 1. Setup

Import the uniform load tool and the metrics watcher. Both read your own endpoint from the environment.

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))
from common import loadtest, ttft_metrics
from common.config import print_settings

settings = print_settings()

## 2. Sweep rising concurrency

Run one summary row per concurrency level. Throughput should climb, then flatten, while TTFT p95 turns up at the knee.

In [ ]:
# Requires a live vLLM endpoint. One row per level.
rows = loadtest.sweep([1, 8, 32, 64, 128], input_tokens=256, output_tokens=128)

**What you should see (TODO, Omer):** throughput rising then flattening, TTFT p95 turning up where the queue forms. The level where throughput stops climbing but latency starts is the knee, your honest capacity.

## 3. Watch saturation live

Drive load in the background and watch the metrics move. Raise concurrency to fill the batch and the queue. Raise `input_tokens` to fill the KV cache.

In [ ]:
# Requires a live vLLM endpoint. Background load + inline metrics.
with loadtest.background(concurrency=64, input_tokens=512, output_tokens=128):
    ttft_metrics.watch(30)

**What you should see (TODO, Omer):** TTFT climbing, KV cache rising toward 100 percent, and `waiting` growing past the knee. If `waiting` spikes while KV is still under 100, the wall is the batch cap, not memory.

## 4. Name the bottleneck

TODO (Omer): read the level where throughput flattened, then ask which signal hit its limit first. KV near 100 with preemptions means KV-cache bound. High waiting with KV low means batch-cap or queue bound. Both flat means compute bound. Du'An tunes the flag that moves it in Module 9.

## Things to know

- TODO (Omer): the knee is the operating point. Past it you pay latency for no extra throughput.
- TODO (Omer): preemption is recompute, not a crash. The server is protecting itself.

## Try it yourself

**TODO (Omer):** lower `output_tokens` and raise concurrency so the queue, not the cache, is the wall. Confirm `waiting` spikes first.

## Summary

- TODO (Omer): mirror the objectives.

## Next

**Hand back to Du'An, Module 9: Tune the Engine.** You found the knee. Next he changes the engine flags that move it, and proves the gain against these same metrics.